# การทำความสะอาดข้อมูลขั้นประยุกต์

เริ่ม Notebook ใหม่ อัปโหลด w03_online_orders_dirty.csv แล้วรันทีละขั้นตอน

- รหัสคำสั่งซื้อเป็นข้อความ ต้องรักษาเลขศูนย์นำหน้า หนึ่ง order_id แทนหนึ่งรายการขาย และชุดฝึกนี้มีเฉพาะแถวซ้ำที่เหมือนกันทุกคอลัมน์
- ค่าว่างที่บันทึกเป็นข้อความ ได้แก่ ข้อความว่าง ช่องว่าง เครื่องหมาย - คำว่า ไม่ระบุ และ N/A ต้องปรับเป็นค่าว่างมาตรฐาน ไม่เดาราคาหรือจำนวนที่ยังไม่ทราบ
- ราคาใช้หน่วยบาทและไม่ติดลบ คำว่า ฟรี ได้รับการยืนยันในกติกาชุดฝึกว่าหมายถึงราคา 0 บาท ส่วนคำว่า สอบถาม ยังไม่ทราบราคา
- วันที่ยอมรับเฉพาะ YYYY-MM-DD และ DD/MM/YYYY ปี ค.ศ. โดย 03/11/2026 หมายถึง 3 พฤศจิกายน 2026 วันที่ไม่มีจริงต้องแยกไว้ตรวจสอบ
- ข้อมูลในคาบนี้เป็นรายการขายที่จำนวนต้องเป็นจำนวนเต็มบวก คำศัพท์ช่องทางชำระเงินที่พบทั้งหมดมีความหมายตามตารางจับคู่คำที่กำหนด
- แยกข้อมูลที่มีคอลัมน์จำเป็นครบออกจากรายการที่ยังมีค่าว่างหลังแปลง โดยเก็บทั้งสองชุดและตรวจสอบจำนวนแถวให้ตรงกันก่อนส่งออก


## ขั้นตอนที่ 1 นำเข้าไลบรารี pandas

สร้าง Notebook ใหม่ แล้ว import pandas เพื่อใช้จัดการข้อมูลในตาราง



In [1]:
import pandas as pd
print(pd.__version__)

2.3.3


ผลลัพธ์อ้างอิง

```text
2.3.3

```

ข้อควรระวัง: เลขรุ่น pandas อาจต่างจากตัวอย่างได้ แต่ต้องนำเข้าไลบรารีสำเร็จก่อนใช้งาน pd

คำถาม: เมื่อเริ่ม Runtime ใหม่ ต้องรันคำสั่งใดก่อนใช้งาน pd

## ขั้นตอนที่ 2 อ่านไฟล์โดยรักษารหัสและค่าว่างในไฟล์ต้นฉบับ

อ่านทุกคอลัมน์เป็นข้อความ และคงข้อความ N/A กับช่องว่างไว้ตรวจสอบเอง



In [2]:
raw = pd.read_csv('w03_online_orders_dirty.csv',
    dtype='string', keep_default_na=False)
print(raw.shape)
raw.head()

(40, 7)
  order_id  order_date product_name quantity unit_price payment_method status
0     0001  2026-11-01                     2    250 บาท        โอนเงิน    ขาย
1     0002  2026-11-02         แฟ้ม        -      99.00         เงินสด    ขาย
2     0003  2026-11-03        สมุด         4    ไม่ระบุ           CASH    ขาย
3     0004  2026-11-04        ปากกา        1    250 บาท            N/A    ขาย
4     0005  2026-11-05         แฟ้ม        2        ฟรี         เงินสด    ขาย

ผลลัพธ์อ้างอิง

```text
(40, 7)
  order_id  order_date product_name quantity unit_price payment_method status
0     0001  2026-11-01                     2    250 บาท        โอนเงิน    ขาย
1     0002  2026-11-02         แฟ้ม        -      99.00         เงินสด    ขาย
2     0003  2026-11-03        สมุด         4    ไม่ระบุ           CASH    ขาย
3     0004  2026-11-04        ปากกา        1    250 บาท            N/A    ขาย
4     0005  2026-11-05         แฟ้ม        2        ฟรี         เงินสด    ขาย
```

ข้อควรระวัง: อัปโหลดไฟล์ w03_online_orders_dirty.csv ก่อนรัน และใช้ตัวพิมพ์ชื่อไฟล์ให้ตรงกัน

คำถาม: รหัส 0001 ยังมีเลขศูนย์นำหน้าหรือไม่

## ขั้นตอนที่ 3 สำรวจค่าที่อาจเป็นค่าว่างก่อนแก้ไข

ใช้ value_counts() รวมค่าที่เหมือนกันและนับจำนวน จึงไม่ต้องอ่านข้อมูลทีละแถว แม้มี 10,000 แถว ข้อความว่างและช่องว่างอาจแสดงเป็นบรรทัดที่ไม่มีข้อความ จึงต้องตรวจความหมายก่อนแทนค่า จากผลลัพธ์ให้เลือกค่าที่น่าสงสัย แล้วตรวจความหมายกับเงื่อนไขของชุดข้อมูลก่อนแก้ไข



In [3]:
print(raw.dtypes)
columns_to_check = ['product_name', 'quantity',
    'unit_price', 'payment_method', 'order_date']
for column in columns_to_check:
    print('\nคอลัมน์:', column)
    print(raw[column].value_counts(dropna=False))

order_id          string[python]
order_date        string[python]
product_name      string[python]
quantity          string[python]
unit_price        string[python]
payment_method    string[python]
status            string[python]
dtype: object

คอลัมน์: product_name
product_name
 สมุด     14
แฟ้ม      13
ปากกา     12
           1
Name: count, dtype: Int64

คอลัมน์: quantity
quantity
2      10
1      10
4       9
3       9
-       1
สอง     1
Name: count, dtype: Int64

คอลัมน์: unit_price
unit_price
250 บาท        13
฿1,250         12
99.00          11
ไม่ระบุ         1
ฟรี             1
สอบถาม          1
 2,450 บาท      1
Name: count, dtype: Int64

คอลัมน์: payment_method
payment_method
CASH       14
เงินสด     13
โอนเงิน    11
N/A         1
QR          1
Name: count, dtype: Int64

คอลัมน์: order_date
order_date
2026-11-01    9
2026-11-04    8
2026-11-02    7
2026-11-03    7
2026-11-05    6
31/02/2026    1
03/11/2026    1
              1
Name: count, dtype: Int64


ผลลัพธ์อ้างอิง

```text
order_id          string[python]
order_date        string[python]
product_name      string[python]
quantity          string[python]
unit_price        string[python]
payment_method    string[python]
status            string[python]
dtype: object

คอลัมน์: product_name
product_name
 สมุด     14
แฟ้ม      13
ปากกา     12
           1
Name: count, dtype: Int64

คอลัมน์: quantity
quantity
2      10
1      10
4       9
3       9
-       1
สอง     1
Name: count, dtype: Int64

คอลัมน์: unit_price
unit_price
250 บาท        13
฿1,250         12
99.00          11
ไม่ระบุ         1
ฟรี             1
สอบถาม          1
 2,450 บาท      1
Name: count, dtype: Int64

คอลัมน์: payment_method
payment_method
CASH       14
เงินสด     13
โอนเงิน    11
N/A         1
QR          1
Name: count, dtype: Int64

คอลัมน์: order_date
order_date
2026-11-01    9
2026-11-04    8
2026-11-02    7
2026-11-03    7
2026-11-05    6
31/02/2026    1
03/11/2026    1
              1
Name: count, dtype: Int64

```

ข้อควรระวัง: รายการที่นับได้เป็นข้อมูลสำหรับตรวจสอบ ไม่ใช่ข้อสรุปว่าค่าใดผิด ชุดนี้ยืนยันว่าข้อความว่าง ช่องว่าง - ไม่ระบุ และ N/A แทนค่าว่าง ส่วน ฟรี หมายถึง 0 บาท และ สอบถาม ยังไม่ทราบราคา หากมีค่าที่ไม่ซ้ำจำนวนมาก ให้ตรวจรูปแบบและเงื่อนไขเฉพาะคอลัมน์เพิ่มเติม เช่น ราคาและวันที่ในขั้นตอนถัดไป ไม่พิจารณาเฉพาะค่าที่พบบ่อย หากผลลัพธ์มี ... แสดงว่ารายการถูกย่อ ไม่ได้แสดงทุกค่า

คำถาม: พบข้อความใดที่อาจแทนค่าว่าง ในคอลัมน์ใด และมีจำนวนเท่าใด เหตุใดคำว่า ฟรี กับ สอบถาม จึงยังไม่ควรถูกแทนเป็น pd.NA ทันที

## ขั้นตอนที่ 4 สร้างสำเนาและตัดช่องว่างทุกคอลัมน์

เก็บ raw ไว้เปรียบเทียบ แล้ววนทำ .str.strip() ให้แต่ละคอลัมน์



In [4]:
df = raw.copy()
for column in df.columns:
    df[column] = df[column].str.strip()
df.head()

  order_id  order_date product_name quantity unit_price payment_method status
0     0001  2026-11-01                     2    250 บาท        โอนเงิน    ขาย
1     0002  2026-11-02         แฟ้ม        -      99.00         เงินสด    ขาย
2     0003  2026-11-03         สมุด        4    ไม่ระบุ           CASH    ขาย
3     0004  2026-11-04        ปากกา        1    250 บาท            N/A    ขาย
4     0005  2026-11-05         แฟ้ม        2        ฟรี         เงินสด    ขาย

ผลลัพธ์อ้างอิง

```text
  order_id  order_date product_name quantity unit_price payment_method status
0     0001  2026-11-01                     2    250 บาท        โอนเงิน    ขาย
1     0002  2026-11-02         แฟ้ม        -      99.00         เงินสด    ขาย
2     0003  2026-11-03         สมุด        4    ไม่ระบุ           CASH    ขาย
3     0004  2026-11-04        ปากกา        1    250 บาท            N/A    ขาย
4     0005  2026-11-05         แฟ้ม        2        ฟรี         เงินสด    ขาย
```

ข้อควรระวัง: บรรทัดภายใน for ต้องเยื้อง 4 ช่อง และอย่าใช้ .strip() ตัดช่องว่างภายในชื่อสินค้าโดยไม่พิจารณาความหมาย

คำถาม: หลังดำเนินการ raw เปลี่ยนตาม df หรือไม่

## ขั้นตอนที่ 5 เปลี่ยนข้อความที่แทนค่าว่างเป็น pd.NA

ใช้ผลสำรวจจากขั้นตอนที่ 3 และเงื่อนไขของชุดข้อมูลยืนยันรายการก่อนแทนค่า: ข้อความว่าง ช่องว่าง - ไม่ระบุ และ N/A หมายถึงไม่มีข้อมูล หลังขั้นตอนที่ 4 ช่องว่างล้วนจะเป็นข้อความว่างแล้ว จึงแทนค่าที่ตกลงไว้ด้วย pd.NA และนับผลอีกครั้ง ไม่เพิ่มคำอื่นในรายการโดยคาดเดา



In [5]:
df = df.replace(['', '-', 'ไม่ระบุ', 'N/A'], pd.NA)
df.isna().sum()

order_id          0
order_date        1
product_name      1
quantity          1
unit_price        1
payment_method    1
status            0

ผลลัพธ์อ้างอิง

```text
order_id          0
order_date        1
product_name      1
quantity          1
unit_price        1
payment_method    1
status            0
```

ข้อควรระวัง: ค่าจำนวน -2 ไม่ใช่ข้อความ - จึงไม่ถูกแทนเป็นค่าว่าง และค่าว่างไม่ได้แปลว่า 0

คำถาม: คอลัมน์ใดมีค่าว่างเพิ่มขึ้นหลังปรับมาตรฐาน

## ขั้นตอนที่ 6 ปรับชื่อช่องทางชำระเงินให้ตรงกัน

รวมเฉพาะคำที่กติกาชุดฝึกยืนยันว่ามีความหมายเดียวกัน



In [6]:
payment_map = {'CASH': 'เงินสด', 'สด': 'เงินสด',
    'โอนเงิน': 'พร้อมเพย์', 'โอน': 'พร้อมเพย์', 'QR': 'พร้อมเพย์'}
df['payment_method'] = df['payment_method'].replace(payment_map)
print(df['payment_method'].value_counts(dropna=False))

payment_method
เงินสด       27
พร้อมเพย์    12
<NA>          1
Name: count, dtype: Int64


ผลลัพธ์อ้างอิง

```text
payment_method
เงินสด       27
พร้อมเพย์    12
<NA>          1
Name: count, dtype: Int64

```

ข้อควรระวัง: คำที่ไม่ทราบความหมายต้องตรวจสอบ ไม่กำหนดให้เป็นหมวดที่มีอยู่โดยอัตโนมัติ

คำถาม: หลังปรับชื่อให้ตรงกันเหลือช่องทางชำระเงินกี่รูปแบบ

## ขั้นตอนที่ 7 ตัดสัญลักษณ์และหน่วยออกจากราคา

ตัดเฉพาะเครื่องหมายที่กำหนด โดยยังไม่เดาราคาที่ไม่ทราบ



In [7]:
price_text = (df['unit_price']
    .str.replace('฿', '', regex=False)
    .str.replace('บาท', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip())
price_text.head(10)

0       250
1     99.00
2      <NA>
3       250
4       ฟรี
5    สอบถาม
6       250
7     99.00
8      1250
9       250

ผลลัพธ์อ้างอิง

```text
0       250
1     99.00
2      <NA>
3       250
4       ฟรี
5    สอบถาม
6       250
7     99.00
8      1250
9       250
```

ข้อควรระวัง: ตรวจผลลัพธ์ให้ตรงกับขั้นตอนก่อนเริ่มคำสั่งถัดไป

คำถาม: ข้อความ ฿1,250 กลายเป็นข้อความใดก่อนแปลงเป็นตัวเลข

## ขั้นตอนที่ 8 แปลงราคาเป็นตัวเลขและตรวจค่าที่แปลงไม่ได้

เป้าหมายคือค้นหารายการที่ราคายังว่างหลังแปลงเป็นตัวเลข เพื่อทราบว่ารายการใดต้องตรวจสอบต่อ โดย pd.to_numeric() แปลงข้อความเป็นตัวเลข จากนั้น .isna() ตรวจว่าราคาในแต่ละแถวว่างหรือไม่ และ .loc ใช้ผลตรวจนั้นเลือกเฉพาะแถวที่ราคาว่าง พร้อมแสดงรหัสคำสั่งซื้อและราคา ผลลัพธ์เป็นตารางย่อยสำหรับตรวจสอบ ไม่ได้ลบแถวออกจาก df

- `pd.to_numeric(price_text, errors='coerce')`: หน้าที่: แปลงข้อความราคาเป็นตัวเลข เช่น ข้อความ 1250 กลายเป็นตัวเลข 1250 ส่วน errors='coerce' ให้ค่าที่แปลงไม่ได้ เช่น สอบถาม กลายเป็นค่าว่าง เพื่อให้ตรวจสอบต่อได้
- `price_text.replace({'ฟรี': '0'})`: หน้าที่: เปลี่ยนคำว่า ฟรี เป็นข้อความ 0 ก่อนแปลงเป็นตัวเลข ตามเงื่อนไขของชุดข้อมูลนี้
- `df['unit_price'].isna()`: หน้าที่: ตรวจราคาแต่ละแถวว่าเป็นค่าว่างหรือไม่ ผลลัพธ์เป็น True หรือ False ตามลำดับแถว เช่น ราคา 250 ได้ False ส่วนราคาว่างได้ True คำสั่งนี้ตรวจอย่างเดียว ยังไม่ได้เลือกหรือลบแถว
- `df.loc[เงื่อนไขเลือกแถว, รายชื่อคอลัมน์]`: หน้าที่: เลือกข้อมูลบางส่วนจากตาราง df ในตัวอย่างนี้ใช้กรองแถวตามผล True/False จาก .isna() โดยเก็บเฉพาะแถวที่ได้ True แล้วเลือกคอลัมน์ที่ต้องการแสดง ผลที่ได้เป็นตารางย่อย ไม่ได้แก้ไขตารางต้นฉบับ
- `['order_id', 'unit_price']`: หน้าที่: ระบุให้ตารางผลลัพธ์แสดงรหัสคำสั่งซื้อกับราคาเท่านั้น เพื่อระบุได้ว่าราคาที่ว่างเป็นของรายการใด
- `df.loc[df['unit_price'].isna(), ['order_id', 'unit_price']]`: อ่านทั้งบรรทัดว่า: จากตาราง df ให้แสดงรหัสคำสั่งซื้อและราคา เฉพาะรายการที่ราคาว่าง ตัวอย่าง หากราคา 3 แถวเป็น 250, ค่าว่าง, 99 จะได้ผลลัพธ์เฉพาะแถวที่สอง พร้อมรหัสคำสั่งซื้อของแถวนั้น



In [8]:
price_text = price_text.replace({'ฟรี': '0'})
df['unit_price'] = pd.to_numeric(price_text, errors='coerce')
df.loc[df['unit_price'].isna(), ['order_id', 'unit_price']]

  order_id  unit_price
2     0003        <NA>
5     0006        <NA>

ผลลัพธ์อ้างอิง

```text
  order_id  unit_price
2     0003        <NA>
5     0006        <NA>
```

ข้อควรระวัง: errors=coerce ไม่ได้แก้ข้อมูลให้ถูกต้อง ต้องนับและติดตามรายการที่แปลงไม่ได้ พร้อมเทียบ raw .loc ในบรรทัดนี้ใช้แสดงข้อมูลเท่านั้น ไม่มีการกำหนดค่าด้วย = จึงไม่ได้แก้ไข df และค่าว่างที่พบอาจมีมาตั้งแต่ต้นหรือเกิดจากการแปลงไม่สำเร็จ

คำถาม: ถ้าเงื่อนไขได้ True, False, True จะเลือกแถวใด และแสดงคอลัมน์ใด คำสั่งนี้ลบแถวออกจาก df หรือไม่

## ขั้นตอนที่ 9 แปลงจำนวนสินค้าเป็นตัวเลข

แปลงข้อความจำนวนเป็นตัวเลข และเก็บค่าที่แปลงไม่ได้เป็นค่าว่างเพื่อแยกตรวจสอบ



In [9]:
df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce')
df[['order_id', 'quantity']].head(20)

   order_id  quantity
0      0001         2
1      0002      <NA>
2      0003         4
3      0004         1
4      0005         2
5      0006         3
6      0007         4
7      0008         1
8      0009         2
9      0010         3
10     0011         4
11     0012         1
12     0013         2
13     0014         3
14     0015         4
15     0016         1
16     0017         2
17     0018         3
18     0019      <NA>
19     0020         1

ผลลัพธ์อ้างอิง

```text
   order_id  quantity
0      0001         2
1      0002      <NA>
2      0003         4
3      0004         1
4      0005         2
5      0006         3
6      0007         4
7      0008         1
8      0009         2
9      0010         3
10     0011         4
11     0012         1
12     0013         2
13     0014         3
14     0015         4
15     0016         1
16     0017         2
17     0018         3
18     0019      <NA>
19     0020         1
```

ข้อควรระวัง: ชุดฝึกนี้ใช้รายการขายที่มีจำนวนเต็มบวกเท่านั้น และไม่กำหนดให้จำนวนที่ไม่ทราบเท่ากับ 1

คำถาม: ข้อความจำนวนที่แปลงไม่ได้ควรเติมเป็น 1 ทันทีหรือไม่

## ขั้นตอนที่ 10 แปลงวันที่รูปแบบ YYYY-MM-DD

แปลงวันที่รูปแบบ YYYY-MM-DD อย่างชัดเจน ค่าที่ไม่ตรงจะเป็น NaT



In [10]:
df['date_clean'] = pd.to_datetime(df['order_date'],
    format='%Y-%m-%d', errors='coerce')
df[['order_id', 'order_date', 'date_clean']].head(10)

  order_id  order_date date_clean
0     0001  2026-11-01 2026-11-01
1     0002  2026-11-02 2026-11-02
2     0003  2026-11-03 2026-11-03
3     0004  2026-11-04 2026-11-04
4     0005  2026-11-05 2026-11-05
5     0006  2026-11-01 2026-11-01
6     0007  31/02/2026        NaT
7     0008  03/11/2026        NaT
8     0009  2026-11-04 2026-11-04
9     0010  2026-11-05 2026-11-05

ผลลัพธ์อ้างอิง

```text
  order_id  order_date date_clean
0     0001  2026-11-01 2026-11-01
1     0002  2026-11-02 2026-11-02
2     0003  2026-11-03 2026-11-03
3     0004  2026-11-04 2026-11-04
4     0005  2026-11-05 2026-11-05
5     0006  2026-11-01 2026-11-01
6     0007  31/02/2026        NaT
7     0008  03/11/2026        NaT
8     0009  2026-11-04 2026-11-04
9     0010  2026-11-05 2026-11-05
```

ข้อควรระวัง: ตรวจผลลัพธ์ให้ตรงกับขั้นตอนก่อนเริ่มคำสั่งถัดไป

คำถาม: วันที่ 03/11/2026 ยังแปลงไม่ได้ในขั้นนี้ เพราะเหตุใด

## ขั้นตอนที่ 11 แปลงวันที่รูปแบบ DD/MM/YYYY

ลองรูปแบบ DD/MM/YYYY สำหรับค่าที่ยังแปลงไม่ได้ แล้วเติมจากวันที่ที่แปลงสำเร็จเท่านั้น

- `format='%d/%m/%Y'`: กำหนดวันก่อนเดือน
- `.fillna(slash_dates)`: ใช้ผลแปลงอีกชุดเฉพาะตำแหน่งที่ว่าง
- `df.loc[df['date_clean'].isna(), ['order_id', 'order_date']]`: หน้าที่: กรองรายการที่ยังแปลงวันที่ไม่ได้ โดย .isna() ตรวจ date_clean แล้ว .loc เลือกเฉพาะแถวที่ได้ True ผลลัพธ์แสดงรหัสกับวันที่ต้นฉบับ เพื่อค้นหาวันที่ที่ต้องตรวจสอบ เช่น 31/02/2026



In [11]:
slash_dates = pd.to_datetime(df['order_date'],
    format='%d/%m/%Y', errors='coerce')
df['date_clean'] = df['date_clean'].fillna(slash_dates)
df.loc[df['date_clean'].isna(), ['order_id', 'order_date']]

   order_id  order_date
6      0007  31/02/2026
14     0015        <NA>

ผลลัพธ์อ้างอิง

```text
   order_id  order_date
6      0007  31/02/2026
14     0015        <NA>
```

ข้อควรระวัง: ห้ามใช้วันที่วันนี้แทนวันที่ไม่มีจริงโดยไม่มีหลักฐาน และวันที่ 03/11/2026 ของชุดนี้หมายถึง 3 พฤศจิกายน

คำถาม: 31/02/2026 ต้องได้วันที่ใดหรือควรแยกตรวจสอบ

## ขั้นตอนที่ 12 ลบแถวซ้ำที่เหมือนกันทุกคอลัมน์

ตรวจและลบแถวซ้ำที่เหมือนกันทุกคอลัมน์ หลังจัดรูปแบบแล้ว



In [12]:
print('แถวซ้ำ:', df.duplicated().sum())
df = df.drop_duplicates().copy()
print('หลังลบแถวซ้ำ:', df.shape)

แถวซ้ำ: 4
หลังลบแถวซ้ำ: (36, 8)


ผลลัพธ์อ้างอิง

```text
แถวซ้ำ: 4
หลังลบแถวซ้ำ: (36, 8)

```

ข้อควรระวัง: ตรวจผลลัพธ์ให้ตรงกับขั้นตอนก่อนเริ่มคำสั่งถัดไป

คำถาม: จำนวนแถวลดลงเท่าใด และลบแถวซ้ำที่เหมือนกันทุกคอลัมน์หรือไม่

## ขั้นตอนที่ 13 ตรวจคุณภาพและแยกรายการที่ยังมีค่าว่าง

ตรวจคอลัมน์จำเป็นหลังแปลงข้อมูล แยกรายการที่ยังไม่ครบไว้ตรวจสอบ และรักษา raw ไว้เปรียบเทียบ

- `required`: รายชื่อคอลัมน์จำเป็นสำหรับใช้งาน
- `.dropna(subset=required)`: เลือกเฉพาะรายการที่มีข้อมูลจำเป็นครบ
- `.isna().any(axis=1)`: ตรวจว่าแถวนั้นมีค่าว่างอย่างน้อยหนึ่งคอลัมน์
- `df.loc[df[required].isna().any(axis=1)]`: หน้าที่: เลือกรายการที่ข้อมูลยังไม่ครบ โดย .isna() ตรวจแต่ละช่อง และ .any(axis=1) รวมผลตามแถว: ถ้ามีค่าว่างอย่างน้อยหนึ่งช่องจะได้ True จากนั้น .loc เลือกแถวนั้นพร้อมทุกคอลัมน์ เพื่อนำไปเก็บใน review
- `review = ... .copy()`: เก็บสำเนาแถวที่เลือกไว้ใน review โดย df ยังมีแถวเหล่านั้นอยู่



In [13]:
required = ['order_id', 'date_clean', 'product_name',
    'quantity', 'unit_price', 'payment_method', 'status']
print(df[required].isna().sum())
ready = df.dropna(subset=required)[required].copy()
review = df.loc[df[required].isna().any(axis=1)].copy()
print('พร้อมใช้งาน:', ready.shape)
print('รอตรวจสอบ:', review.shape)

order_id          0
date_clean        2
product_name      1
quantity          2
unit_price        2
payment_method    1
status            0
dtype: int64
พร้อมใช้งาน: (28, 7)
รอตรวจสอบ: (8, 8)


ผลลัพธ์อ้างอิง

```text
order_id          0
date_clean        2
product_name      1
quantity          2
unit_price        2
payment_method    1
status            0
dtype: int64
พร้อมใช้งาน: (28, 7)
รอตรวจสอบ: (8, 8)

```

ข้อควรระวัง: จำนวนแถวที่ครบไม่ได้ยืนยันความถูกต้องทุกด้าน ชุดฝึกนี้กำหนดว่ารายการที่มีค่าครบจะเป็นไปตามกติกา ส่วนค่าว่างต้องตรวจหลักฐานก่อนเติม

คำถาม: ready รวมกับ review เท่ากับจำนวนแถวหลังลบแถวซ้ำหรือไม่

## ขั้นตอนที่ 14 Export ไฟล์ CSV

บันทึกข้อมูลเป็นไฟล์ CSV สองไฟล์: ready สำหรับข้อมูลพร้อมใช้งาน และ review สำหรับรายการรอตรวจสอบ จากนั้นดาวน์โหลดผ่านแถบ Files ใน Google Colab



In [14]:
ready.to_csv('w03_orders_ready.csv',
    index=False, encoding='utf-8-sig')
review.to_csv('w03_orders_review.csv',
    index=False, encoding='utf-8-sig')

ผลลัพธ์อ้างอิง

```text

```

ข้อควรระวัง: เมื่อรันสำเร็จ คำสั่ง to_csv() จะไม่แสดงตารางผลลัพธ์ ให้กดรีเฟรชที่แถบ Files แล้วดาวน์โหลดไฟล์ CSV ทั้งสองไฟล์

คำถาม: ไฟล์ใดเก็บข้อมูลพร้อมใช้งาน และไฟล์ใดเก็บรายการรอตรวจสอบ